In [1]:
import os
from dotenv import load_dotenv
from datetime import date, timedelta
import requests
import zipfile
import io
import logging
from datetime import date, timedelta
import pandas as pd
import numpy as np
import yfinance as yfinance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from edinet_xbrl.edinet_xbrl_parser import EdinetXbrlParser

load_dotenv()
EDINET_API_KEY = os.getenv('EDINET_API_KEY')

In [2]:
def get_documents_by_date(target_date, doc_type='030000'):
    """
    指定した日付にEDINETで開示された書類一覧を取得し、
    指定doc_type(有報)を満たすdoc_idとedinet_codeのリストを返す。
    """
    if isinstance(target_date, date):
        date_str = target_date.strftime("%Y-%m-%d")
    else:
        date_str = target_date
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents.json"
    params = {'date': date_str, 'type': 2,'Subscription-Key':EDINET_API_KEY}
    r = requests.get(base_url, params=params)
    r.raise_for_status()
    data = r.json()
    results = data.get('results', [])

    docs = []


    for d in results:
        if d.get('formCode') == doc_type and d.get('docTypeCode') == '120':
            docs.append(d)

    return docs

h = get_documents_by_date(date(2024, 6, 11),doc_type='030000')

In [11]:
doc_id = h[0]['docID']
periodEnd_date= date.fromisoformat(h[0]['periodEnd'])

In [55]:
h

[{'seqNumber': 132,
  'docID': 'S100TKQK',
  'edinetCode': 'E03575',
  'secCode': '83660',
  'JCN': '6160001000993',
  'filerName': '株式会社滋賀銀行',
  'fundCode': None,
  'ordinanceCode': '010',
  'formCode': '030000',
  'docTypeCode': '120',
  'periodStart': '2023-04-01',
  'periodEnd': '2024-03-31',
  'submitDateTime': '2024-06-11 10:05',
  'docDescription': '有価証券報告書－第137期(2023/04/01－2024/03/31)',
  'issuerEdinetCode': None,
  'subjectEdinetCode': None,
  'subsidiaryEdinetCode': None,
  'currentReportReason': None,
  'parentDocID': None,
  'opeDateTime': None,
  'withdrawalStatus': '0',
  'docInfoEditStatus': '0',
  'disclosureStatus': '0',
  'xbrlFlag': '1',
  'pdfFlag': '1',
  'attachDocFlag': '1',
  'englishDocFlag': '0',
  'csvFlag': '1',
  'legalStatus': '1'}]

In [12]:
import os
import io
import zipfile
import tempfile
import requests
import logging

def download_xbrl_file(doc_id,output_dir = "./xbrl/"):
    """

    """
    path = output_dir + doc_id + '/'
    
    base_url = "https://disclosure.edinet-fsa.go.jp/api/v2/documents"
    params = {
        'type': 1,
        'Subscription-Key': EDINET_API_KEY
    }
    
    try:
        response = requests.get(f"{base_url}/{doc_id}", params=params, stream=True)
        response.raise_for_status()
    except Exception as e:
        logging.error("EDINET APIからのダウンロードに失敗しました: %s", e)
        return None
    
    # try:
    # ダウンロードしたZIPアーカイブをメモリ上で読み込む
    
    z = zipfile.ZipFile(io.BytesIO(response.content))
    # 出力先ディレクトリの作成
    if not os.path.exists(path):
        os.makedirs(path)
            
    filename = doc_id + ".zip"
    with open(path+filename, 'wb') as f:    
        for chunk in response.iter_content(chunk_size=1024):
          f.write(chunk)

    with zipfile.ZipFile(path+filename) as zip_f:
        zip_f.extractall(path)


        # ZIP 内で拡張子が .xbrl のファイルを検索（大文字小文字区別しない）
        xbrl_files = [f for f in zip_f.namelist() if f.lower().endswith('.xbrl')]
        if not xbrl_files:
            logging.warning("doc_id=%s のZIP内にXBRLファイルが見つかりませんでした", doc_id)
            return None
        
        # 例として最初に見つかった XBRL ファイルの絶対パスを返す
        xbrl_file_rel_path = xbrl_files[0]
        current_directory = os.getcwd()
        xbrl_file_abs_path = os.path.join(current_directory + path[1:], xbrl_file_rel_path)
        
    return xbrl_file_abs_path
    

In [13]:
xbrl_file_abs_path = download_xbrl_file(doc_id)

In [37]:
import os
import pandas as pd
from arelle import Cntlr

def extract_financial_data(xbrl_file):
    # Arelle のコントローラ作成（ログ出力は標準出力に設定）
    cntlr = Cntlr.Cntlr(logFileName='logToPrint')
    
    # XBRL ファイルを読み込み
    modelXbrl = cntlr.modelManager.load(xbrl_file)
    
    # 抽出データを格納するリスト
    data = []
    
    # 各 fact から情報を抽出
    for fact in modelXbrl.facts:
        # 必要な情報例: コンセプト、値、単位、コンテキストID
        row = {
            'concept': fact.concept.qname.localName,
            'concept_jp':fact.concept.label(preferredLabel=None, lang='ja', linkroleHint=None), 
            'value': fact.value,
            'unit': fact.unitID if fact.unitID else '',
            'context': fact.contextID,

        }
        data.append(row)
    
    # リストを pandas DataFrame に変換
    df = pd.DataFrame(data)
    return df

def set_context_YYYY(context_str):
    """
    context の文字列に含まれるキーワードに基づいて、
    対象期を periodEnd から何年前か計算し、'YYYY' 形式で返す。
    """
    if "CurrentYear" in context_str:
        target_date = periodEnd_date
    elif "Prior1Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 1)
    elif "Prior2Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 2)
    elif "Prior3Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 3)
    elif "Prior4Year" in context_str:
        target_date = periodEnd_date.replace(year=periodEnd_date.year - 4)
    else:
        # 対象外の場合は None を返す
        return None
    
    return target_date.strftime("%Y")

def set_context_NCM(context_str):
    """
    context の文字列に含まれるキーワードに基づいて、
    連結か非連結（単独）かを判断する。
    """
    if "NonConsolidatedMember" in context_str:
        return True
        
    return False


xbrl_path = xbrl_file_abs_path  # 解析対象の XBRL ファイルパスに変更
if os.path.exists(xbrl_path):
    df = extract_financial_data(xbrl_path)

else:
    print("XBRL ファイルが見つかりません:", xbrl_path)


In [57]:
df

,concept,concept_jp,value,unit,context,context_YYYY,context_NCM
0,NumberOfSubmissionDEI,提出回数,1,pure,FilingDateInstant,None,False
1,OrdinaryIncomeSummaryOfBusinessResults,経常収益,88871000000,JPY,Prior4YearDuration,2020,False
2,OrdinaryIncomeSummaryOfBusinessResults,経常収益,85715000000,JPY,Prior3YearDuration,2021,False
3,OrdinaryIncomeSummaryOfBusinessResults,経常収益,98306000000,JPY,Prior2YearDuration,2022,False
4,OrdinaryIncomeSummaryOfBusinessResults,経常収益,115289000000,JPY,Prior1YearDuration,2023,False
...,...,...,...,...,...,...,...
2248,OtherInformationFinancialStatementsEtcTextBlock,その他,"<p class=""smt_head3"" style=""orphans:0;widows:0...",,CurrentYearDuration,2024,False
2249,OverviewOfOperationalProceduresForSharesTextBlock,提出会社の株式事務の概要,"\n<h2 class=""smt_head1"">第６ 【提出会社の株式事務の概要】</h2>...",,FilingDateInstant,None,False
2250,InformationAboutParentCompanyEtcOfReportingCom...,提出会社の親会社等の情報,\n１ 【提出会社の親会社等の情報】当行は、法第24条の７第１項に規定する親会社等はございま...,,FilingDateInstant,None,False
2251,OtherReferenceInformationTextBlock,その他の参考情報,"\n<h3 class=""smt_head2"" style=""font-family:&ap...",,FilingDateInstant,None,False


### assets = df[df['unit']=='JPY']
# assets = assets.copy()
# assets['context_YYYY'] = assets['context'].apply(set_context_YYYY)
# assets['context_NCM'] = assets['context'].apply(set_context_NCM)

df = df.copy()
df['context_YYYY'] = df['context'].apply(set_context_YYYY)
df['context_NCM'] = df['context'].apply(set_context_NCM)

2250    \n１ 【提出会社の親会社等の情報】当行は、法第24条の７第１項に規定する親会社等はございま...
Name: value, dtype: object

In [53]:
columns_to_check = ['concept_jp']

filtered_df = df[
    df[columns_to_check]
    .apply(lambda row: row.astype(str).str.contains("配当").any(), axis=1)
]

In [56]:
filtered_df

,concept,concept_jp,value,unit,context,context_YYYY,context_NCM
161,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,40.00,JPYPerShares,Prior4YearDuration_NonConsolidatedMember,2020,True
162,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,40.00,JPYPerShares,Prior3YearDuration_NonConsolidatedMember,2021,True
163,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,80.00,JPYPerShares,Prior2YearDuration_NonConsolidatedMember,2022,True
164,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,80.00,JPYPerShares,Prior1YearDuration_NonConsolidatedMember,2023,True
165,DividendPaidPerShareSummaryOfBusinessResults,１株当たり配当額,90.00,JPYPerShares,CurrentYearDuration_NonConsolidatedMember,2024,True
166,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior4YearDuration_NonConsolidatedMember,2020,True
167,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior3YearDuration_NonConsolidatedMember,2021,True
168,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,17.50,JPYPerShares,Prior2YearDuration_NonConsolidatedMember,2022,True
169,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,40.00,JPYPerShares,Prior1YearDuration_NonConsolidatedMember,2023,True
170,InterimDividendPaidPerShareSummaryOfBusinessRe...,１株当たり中間配当額,50.00,JPYPerShares,CurrentYearDuration_NonConsolidatedMember,2024,True


In [73]:
from dotenv import load_dotenv
import os
from sshtunnel import SSHTunnelForwarder
from pymongo import MongoClient


# .envファイルをロードして環境変数を読み込む
load_dotenv()

# SSH接続情報（.envから読み込み）
SSH_HOST = os.getenv('SSH_HOST')
SSH_PORT = int(os.getenv('SSH_PORT', 22))
SSH_USERNAME = os.getenv('SSH_USERNAME')
SSH_PASSWORD = os.getenv('SSH_PASSWORD')

# MongoDB接続情報（.envから読み込み）
MONGO_HOST = os.getenv('MONGO_HOST', '127.0.0.1')
MONGO_PORT = int(os.getenv('MONGO_PORT', 27017))
DATABASE_NAME = 'test'
COLLECTION_NAME = 'collection'
    
def ssh_connection_and_write(df):
    tunnel = SSHTunnelForwarder(
        (SSH_HOST, SSH_PORT),
        ssh_username=SSH_USERNAME,
        ssh_password=SSH_PASSWORD,
        remote_bind_address=(MONGO_HOST, MONGO_PORT)
    )
    tunnel.start()
    try:
        local_port = tunnel.local_bind_port
        print(f"SSHトンネル確立: ローカルポート {local_port} がリモートの {MONGO_HOST}:{MONGO_PORT} にマッピングされました。")
        
        client = MongoClient('127.0.0.1', local_port)
        db = client[DATABASE_NAME]
        collection = db[COLLECTION_NAME]
        
        records = df.to_dict(orient='records')
        result = collection.insert_many(records)

    finally:
        tunnel.stop()
        
    return result.inserted_ids

ssh_connection_and_write(df)

SSHトンネル確立: ローカルポート 58551 がリモートの 127.0.0.1:27017 にマッピングされました。


[ObjectId('67b7f32f8d297929dd94aee1'),
 ObjectId('67b7f32f8d297929dd94aee2'),
 ObjectId('67b7f32f8d297929dd94aee3'),
 ObjectId('67b7f32f8d297929dd94aee4'),
 ObjectId('67b7f32f8d297929dd94aee5'),
 ObjectId('67b7f32f8d297929dd94aee6'),
 ObjectId('67b7f32f8d297929dd94aee7'),
 ObjectId('67b7f32f8d297929dd94aee8'),
 ObjectId('67b7f32f8d297929dd94aee9'),
 ObjectId('67b7f32f8d297929dd94aeea'),
 ObjectId('67b7f32f8d297929dd94aeeb'),
 ObjectId('67b7f32f8d297929dd94aeec'),
 ObjectId('67b7f32f8d297929dd94aeed'),
 ObjectId('67b7f32f8d297929dd94aeee'),
 ObjectId('67b7f32f8d297929dd94aeef'),
 ObjectId('67b7f32f8d297929dd94aef0'),
 ObjectId('67b7f32f8d297929dd94aef1'),
 ObjectId('67b7f32f8d297929dd94aef2'),
 ObjectId('67b7f32f8d297929dd94aef3'),
 ObjectId('67b7f32f8d297929dd94aef4'),
 ObjectId('67b7f32f8d297929dd94aef5'),
 ObjectId('67b7f32f8d297929dd94aef6'),
 ObjectId('67b7f32f8d297929dd94aef7'),
 ObjectId('67b7f32f8d297929dd94aef8'),
 ObjectId('67b7f32f8d297929dd94aef9'),
 ObjectId('67b7f32f8d2979

In [62]:
records = df.to_dict(orient='records')
records

[{'concept': 'NumberOfSubmissionDEI',
  'concept_jp': '提出回数',
  'value': '1',
  'unit': 'pure',
  'context': 'FilingDateInstant',
  'context_YYYY': None,
  'context_NCM': False},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '88871000000',
  'unit': 'JPY',
  'context': 'Prior4YearDuration',
  'context_YYYY': '2020',
  'context_NCM': False},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '85715000000',
  'unit': 'JPY',
  'context': 'Prior3YearDuration',
  'context_YYYY': '2021',
  'context_NCM': False},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '98306000000',
  'unit': 'JPY',
  'context': 'Prior2YearDuration',
  'context_YYYY': '2022',
  'context_NCM': False},
 {'concept': 'OrdinaryIncomeSummaryOfBusinessResults',
  'concept_jp': '経常収益',
  'value': '115289000000',
  'unit': 'JPY',
  'context': 'Prior1YearDuration',
  'context_YYYY': '2023',
  'context_NC